In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .config("spark.jars.packages", "graphframes:graphframes:0.8.4-spark3.5-s_2.13")
        .getOrCreate()
    )
    return spark


trec_root = Path("~/scratch/trec-tot-2025").expanduser()
parquet_root = trec_root / "data/enwiki/parquet"
processed_root = trec_root / "data/enwiki/processed"
spark = get_spark()

page = spark.read.parquet((parquet_root / "page").as_posix())
pagelinks = spark.read.parquet((parquet_root / "pagelinks").as_posix())
categorylinks = spark.read.parquet((parquet_root / "categorylinks").as_posix())

page.printSchema()
pagelinks.printSchema()
categorylinks.printSchema()

:: loading settings :: url = jar:file:/storage/scratch1/8/amiyaguchi3/trec-tot-2025/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /storage/home/hcoda1/8/amiyaguchi3/.ivy2.5.2/cache
The jars for the packages stored in: /storage/home/hcoda1/8/amiyaguchi3/.ivy2.5.2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b9b9500c-3d37-47cc-ade4-2038a92a5061;1.0
	confs: [default]
	found graphframes#graphframes;0.8.4-spark3.5-s_2.13 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
downloading https://repos.spark-packages.org/graphframes/graphframes/0.8.4-spark3.5-s_2.13/graphframes-0.8.4-spark3.5-s_2.13.jar ...
	[SUCCESSFUL ] graphframes#graphframes;0.8.4-spark3.5-s_2.13!graphframes.jar (633ms)
:: resolution report :: resolve 1211ms :: artifacts dl 652ms
	:: modules in use:
	graphframes#graphframes;0.8.4-spark3.5-s_2.13 from spark-

root
 |-- page_id: integer (nullable = true)
 |-- page_namespace: integer (nullable = true)
 |-- page_title: string (nullable = true)
 |-- page_is_redirect: string (nullable = true)
 |-- page_is_new: string (nullable = true)
 |-- page_random: string (nullable = true)
 |-- page_touched: string (nullable = true)
 |-- page_links_updated: string (nullable = true)
 |-- page_latest: string (nullable = true)
 |-- page_len: string (nullable = true)
 |-- page_content_model: string (nullable = true)
 |-- page_lang: string (nullable = true)

root
 |-- pl_from: integer (nullable = true)
 |-- pl_from_namespace: integer (nullable = true)
 |-- pl_target_id: integer (nullable = true)

root
 |-- cl_from: integer (nullable = true)
 |-- cl_to: string (nullable = true)
 |-- cl_sortkey: string (nullable = true)
 |-- cl_timestamp: string (nullable = true)
 |-- cl_sortkey_prefix: string (nullable = true)
 |-- cl_collation: string (nullable = true)
 |-- cl_type: string (nullable = true)
 |-- cl_collation_id: 

In [2]:
page.count(), pagelinks.count(), categorylinks.count()

(63415160, 1617680710, 205019381)

In [4]:
from graphframes import GraphFrame

v = page.withColumn("id", F.col("page_id"))
e = pagelinks.select(F.col("pl_from").alias("src"), F.col("pl_target_id").alias("dst"))
g = GraphFrame(v, e)
g

GraphFrame(v:[id: int, page_id: int ... 11 more fields], e:[src: int, dst: int])

In [6]:
g.inDegrees.describe().show()

+-------+--------------------+------------------+
|summary|                  id|          inDegree|
+-------+--------------------+------------------+
|  count|            73504053|          73504053|
|   mean| 5.413000962110447E7|22.008047773909826|
| stddev|3.1340407592261635E7| 3014.279247975729|
|    min|                   1|                 1|
|    max|           111058793|          11866386|
+-------+--------------------+------------------+



Let's try a few things.
First we look at the one hop network using the article namespace with no-redirects and double check that it looks like our original graph.
Then we can look at the two hop network and see whether this improves connectivity in the projected graph.

In [8]:
onehop_df = (
    g.find("(a)-[e]->(b)")
    .where("a.id != b.id")
    .where('a.page_namespace = 0 and a.page_is_redirect = "0"')
    .where('b.page_namespace = 0 and b.page_is_redirect = "0"')
)
onehop_df.printSchema()

onehop = GraphFrame(g.vertices, onehop_df.select("e.*")).dropIsolatedVertices().cache()
# how many vertices and how many edges?
display(onehop.vertices.count(), onehop.edges.count())
onehop.inDegrees.describe().show()

root
 |-- a: struct (nullable = false)
 |    |-- page_id: integer (nullable = true)
 |    |-- page_namespace: integer (nullable = true)
 |    |-- page_title: string (nullable = true)
 |    |-- page_is_redirect: string (nullable = true)
 |    |-- page_is_new: string (nullable = true)
 |    |-- page_random: string (nullable = true)
 |    |-- page_touched: string (nullable = true)
 |    |-- page_links_updated: string (nullable = true)
 |    |-- page_latest: string (nullable = true)
 |    |-- page_len: string (nullable = true)
 |    |-- page_content_model: string (nullable = true)
 |    |-- page_lang: string (nullable = true)
 |    |-- id: integer (nullable = true)
 |-- e: struct (nullable = false)
 |    |-- src: integer (nullable = true)
 |    |-- dst: integer (nullable = true)
 |-- b: struct (nullable = false)
 |    |-- page_id: integer (nullable = true)
 |    |-- page_namespace: integer (nullable = true)
 |    |-- page_title: string (nullable = true)
 |    |-- page_is_redirect: string (

6956960

147935217

+-------+--------------------+------------------+
|summary|                  id|          inDegree|
+-------+--------------------+------------------+
|  count|             1917410|           1917410|
|   mean| 2.169298108996928E7| 77.15366927261253|
| stddev|2.1386030686687443E7|1884.4417584637918|
|    min|                  39|                 1|
|    max|            80328808|           1581985|
+-------+--------------------+------------------+



In [9]:
onehop.unpersist()

GraphFrame(v:[id: int, page_id: int ... 11 more fields], e:[src: int, dst: int])

In [ ]:
twohop_df = (
    g.find("(a)-[]->(b)->[]->(c);")
    .where("a.id != c.id")
    .where('a.page_namespace = 0 and a.page_is_redirect = "0"')
    .where("b.page_namespace != 0")
    .where('c.page_namespace = 0 and c.page_is_redirect = "0"')
)